In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

model_name = "llama3-i-1b"
output_dir = Path(f"../output")

## Hit@10

In [ ]:
results = {}
with open(output_dir / model_name / "baseline" / "circuits" / "results.json") as f:
    baseline = json.load(f)

results["epoch_0"] = baseline["mean"]["hit_at_10"]

In [ ]:
exp_name = "VFT"
for folder in (output_dir / model_name / exp_name).iterdir():
    if not folder.is_dir() or not folder.name.startswith("epoch"):
        continue
    with open(folder / "circuits" / "results.json") as f:
        res = json.load(f)
    
    results[folder.name] = res["mean"]["hit_at_10"]

In [ ]:
sorted_results = sorted(results.items(), key=lambda x: int(x[0].split("_")[1]))

In [ ]:
import matplotlib.pyplot as plt

epochs = [int(x[0].split("_")[1]) for x in sorted_results]
hit_at_10 = [x[1] for x in sorted_results]

plt.figure(figsize=(10, 6))
plt.plot(epochs, hit_at_10, marker='o')
plt.xlabel('Epoch')
plt.ylabel('Hit@10')
plt.ylim(0, 1)
plt.title('Hit@10 vs Epoch')
plt.grid(True)
plt.show()

## Jaccard Similarity

In [ ]:
def compute_jaccard_similarity(set1, set2):
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    if union == 0:
        return 0.0
    return intersection / union

In [ ]:
from eap.graph import Graph

g = Graph.from_pt(
    output_dir / model_name / "baseline" / "circuits" / "knowledge_circuit.pt"
)

In [ ]:
baseline_edges = {e_name for e_name, edge in g.edges.items() if edge.in_graph}
baseline_nodes = {n_name for n_name, node in g.nodes.items() if node.in_graph}

In [ ]:
from tqdm import tqdm

exp_name = "VFT"

jaccard_for_epoch = {}
for folder in tqdm((output_dir / model_name / exp_name).iterdir(), desc="Computing Jaccard similarities"):
    if not folder.is_dir() or not folder.name.startswith("epoch"):
        continue
    g_epoch = Graph.from_pt(
        folder / "circuits" / "knowledge_circuit.pt"
    )
    epoch_edges = {e_name for e_name, edge in g_epoch.edges.items() if edge.in_graph}
    epoch_nodes = {n_name for n_name, node in g_epoch.nodes.items() if node.in_graph}
    
    edge_jaccard = compute_jaccard_similarity(baseline_edges, epoch_edges)
    node_jaccard = compute_jaccard_similarity(baseline_nodes, epoch_nodes)
    
    jaccard_for_epoch[folder.name] = {
        "edge_jaccard": edge_jaccard,
        "node_jaccard": node_jaccard
    }

In [ ]:
# prepare lists in epoch order (assume epoch_0 is the baseline -> jaccard 1.0)
edge_jaccards = []
node_jaccards = []
sorted_epochs = sorted(jaccard_for_epoch.keys(), key=lambda x: int(x.split("_")[1]))
epochs = []
for ep in sorted_epochs:
    epochs.append(int(ep.split("_")[1]))
    vals = jaccard_for_epoch.get(ep)
    edge_jaccards.append(vals["edge_jaccard"])
    node_jaccards.append(vals["node_jaccard"])

# plot two graphs side by side: edges and nodes
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(epochs, edge_jaccards, marker="o")
ax1.set_xlabel("Epoch")
ax1.set_xticks(epochs)
ax1.set_ylabel("Jaccard Similarity")
ax1.set_title("Edge Jaccard at each Epoch (w.r.t Epoch 0)")
ax1.grid(True)

ax2.plot(epochs, node_jaccards, marker="o", color="tab:orange")
ax2.set_xlabel("Epoch")
ax2.set_xticks(epochs)
ax2.set_ylabel("Jaccard Similarity")
ax2.set_title("Node Jaccard at each Epoch (w.r.t Epoch 0)")
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.ticker import FuncFormatter

plt.style.use("seaborn-v0_8-whitegrid")

edge_jaccards = []
sorted_epochs = sorted(jaccard_for_epoch.keys(), key=lambda x: int(x.split("_")[1]))
epochs = []
for ep in sorted_epochs:
    epochs.append(int(ep.split("_")[1]))
    vals = jaccard_for_epoch.get(ep)
    edge_jaccards.append(vals["edge_jaccard"])

plt.plot(epochs, edge_jaccards, marker="o")
plt.title("Edges Jaccard Similarity at each Epoch", fontsize=24, fontweight="bold", pad=20)
plt.xlabel("Epoch (#)", fontsize=22, fontweight="semibold", labelpad=15)
plt.xticks(epochs)
plt.ylabel("Jaccard Similarity", fontsize=22, fontweight="semibold", labelpad=15)
#plt.yticks(np.arange(0.41, 0.45, 0.01))
#plt.ylim(0.405, 0.445)

plt.grid(True)
plt.tick_params(axis="both", labelsize=19)
plt.grid(axis="both", linestyle="--", alpha=1)

plt.savefig("circuits_jaccard_similarity.pdf", bbox_inches="tight")

## Testing different circuits

In [ ]:
results = {
    "pre-trained": {},
    "last": {},
    "best": {}
}

metric = "hit_at_10"
exp_name = "VFT"

with open(output_dir / model_name / "baseline" / "circuits" / "results.json") as f:
    baseline = json.load(f)

results["pre-trained"]["epoch_0"] = baseline["mean"][metric]

with open(output_dir / model_name / exp_name / "epoch_10" / "circuits" / "results.json") as f:
    last = json.load(f)

results["last"]["epoch_10"] = last["mean"][metric]

with open(output_dir / model_name / exp_name / "epoch_7" / "circuits" / "results.json") as f:
    best = json.load(f)

results["best"]["epoch_7"] = best["mean"][metric]

In [ ]:
# add baseline
with open(output_dir / model_name / "baseline" / "circuits" / "last.json") as f:
    baseline_last = json.load(f)
results["last"]["epoch_0"] = baseline_last[metric]

with open(output_dir / model_name / "baseline" / "circuits" / "best.json") as f:
    baseline_best = json.load(f)
results["best"]["epoch_0"] = baseline_best[metric]

In [ ]:

for folder in (output_dir / model_name / exp_name).iterdir():
    if not folder.is_dir() or not folder.name.startswith("epoch"):
        continue
    for method in results.keys():
        file_path = folder / "circuits" / f"{method}.json"
        if file_path.exists():
            res = json.load(open(file_path))
            results[method][folder.name] = res[metric]
        

In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")

methods = ["pre-trained", "last", "best"]
# collect all epochs present across methods, sorted numerically
epochs_all = sorted({int(k.split("_")[1]) for m in results.values() for k in m.keys()})

# plotting styles for each method
styles = {
    "pre-trained": {"color": "C0", "marker": "o"},
    "last": {"color": "C1", "marker": "s"},
    "best": {"color": "C2", "marker": "^"},
}

metric_mapping = {
    "hit_at_10": "Hit@10",
}

labels_mapping = {
    "pre-trained": "Before",
    "last": "After",
    "best": "Best",
}

# grouped bar plot for methods across epochs
width = 0.22
x = np.arange(len(epochs_all))
offsets = np.linspace(-width, width, len(methods))

plt.figure(figsize=(10, 6))
for i, method in enumerate(methods):
    y = [results[method].get(f"epoch_{e}", np.nan) for e in epochs_all]
    plt.bar(x + offsets[i], y, width=width, label=labels_mapping[method], color=styles[method]["color"], alpha=0.9, edgecolor="black")

plt.xlabel("Epoch (#)", fontsize=22, fontweight="semibold", labelpad=15)
plt.ylabel(metric_mapping.get(metric, metric), fontsize=22, fontweight="semibold", labelpad=15)
#plt.title(f"{metric_mapping.get(metric, metric)} vs Epoch by method (bar plot)")
plt.title(f"Circuits performance at each Epoch ({metric_mapping.get(metric, metric)})", fontsize=24, fontweight="semibold", pad=20)
plt.xticks(x, epochs_all)
plt.tick_params(axis="both", labelsize=19)
plt.ylim(0.25, 0.7)
plt.grid(axis="y", linestyle="--", alpha=1)
plt.grid(axis="x", alpha=0)
plt.legend(frameon=True, fontsize=18)
plt.tight_layout()
plt.savefig("circuits_performance_epochs.pdf")
plt.show()